# Train BPR + CLIP multimodal hybrid (late fusion)

The mean-pool notebooks (`train-mean-pool-title.ipynb`, `-image.ipynb`, `-clip-multimodal.ipynb`)
have no learned collaborative-filtering signal at all — they're pure content baselines. BPR has
real CF signal but, per `train-bpr2.ipynb`, scores cold items (< 5 train reviews) at exactly
**0.0000** ndcg@20 — an untrained item embedding carries no information regardless of the user.

This notebook **trains a real BPR model** (trainable user/item embeddings, pairwise BPR loss,
gradient descent — collaborative signal) and **blends its score with a frozen content score**
(the same CLIP text+image fused embedding from `train-mean-pool-clip-multimodal.ipynb`) at
prediction time:

```
final_score = ALPHA * normalize(BPR_score) + (1 - ALPHA) * normalize(CLIP_content_score)
```

For a cold item, `BPR_score` is close to noise (untrained embedding) but `CLIP_content_score`
is still meaningful (the item has a title and maybe a photo regardless of purchase history) —
so the blended score should degrade more gracefully than plain BPR's hard 0.0000.

**Why normalize before blending:** BPR dot products are unbounded and grow with training;
CLIP cosine similarities are bounded in [-1, 1]. Without rescaling, a fixed `ALPHA` would let
whichever signal has the larger magnitude dominate regardless of the chosen weight. We
z-score each score vector (per user, across the candidate item set) before blending — a
standard rank-fusion trick that needs no extra training.

**Coverage note:** as of this notebook's creation, only 2 sample CLIP-image parquet shards
exist locally — running here is a **pipeline smoke test**, not a real result.

In [1]:
# ! pip install "pandas<=2.3.2" "numpy" "torch<=2.5" "matplotlib" "seaborn" "matplotlib-venn" "datasets" "ipykernel" "recbole" "kmeans-pytorch" "sentence-transformers" "pyarrow"

In [2]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

import logging
logging.getLogger().handlers.clear()

In [3]:
from typing import Any
import glob
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.abstract_recommender import GeneralRecommender
from recbole.model.loss import BPRLoss
from recbole.model.init import xavier_normal_initialization
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger, InputType, ModelType
from sentence_transformers import SentenceTransformer

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# --- Config ---
DATASET_NAME: str = "beauty"
DATA_DIR: str = "../data"
EMBEDDINGS_DIR: str = "../data/embeddings"
SEED = 67
DEVICE = "mps"  # Other options: "mps", "cuda"
EMBEDDING_SIZE = 64  # trainable BPR embedding size
CLIP_DIM = 512
ALPHA = 0.5  # weight on the (normalized) BPR score; (1 - ALPHA) goes to the CLIP content score

## Create dataset

In [5]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id"],
        "user": ["user_id", "cold"],
        "item": ["item_id", "title", "cold"],
    },
    "embedding_size": EMBEDDING_SIZE,
    "epochs": 100,
    "train_batch_size": 1024,
    "eval_batch_size": 409_600_000,
    "eval_args": {
        "split": None,
        "order": "TO",
        "mode": {"valid": "full", "test": "full"},
    },
    "metrics": ["NDCG", "Recall", "MRR"],
    "topk": [20],
    "valid_metric": "NDCG@20",
    "seed": SEED,
}

# Borrow BPR's defaults (pairwise input type + negative sampling config) for this Config
# object, even though we construct our own model class below instead of RecBole's BPR.
config: Config = Config(model="BPR", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

In [6]:
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work becaus

## CLIP-encode titles and fuse with precomputed image embeddings

Titles are encoded with CLIP's text tower directly. Image embeddings are loaded from
`clip_image_embeddings.pt` (built by the preprocessing notebook), re-indexed from
preprocessing iid to RecBole internal ID, then averaged with the text embeddings and
L2-normalized. Items without an image keep only their title embedding.

In [7]:
logging.getLogger('httpx').setLevel(logging.WARNING)

title_tokens: torch.Tensor = dataset.item_feat["title"]
id2token: dict[int, str] = {v: k for k, v in dataset.field2token_id["title"].items()}
titles: list[str] = [id2token.get(tok.item(), "") for tok in title_tokens]

print(f"Encoding {len(titles):,} item titles with CLIP text tower (clip-ViT-B-32)...")
clip_model = SentenceTransformer("clip-ViT-B-32", device=str(config["device"]))
clip_text_embs = clip_model.encode(titles, show_progress_bar=True, batch_size=256, normalize_embeddings=True)
clip_text_embs = torch.from_numpy(clip_text_embs).float()
print(f"CLIP text embeddings shape: {tuple(clip_text_embs.shape)}")

Encoding 250,853 item titles with CLIP text tower (clip-ViT-B-32)...


23 Jun 12:51    INFO  Loading SentenceTransformer model from sentence-transformers/clip-ViT-B-32.
Batches: 100%|██████████| 980/980 [04:36<00:00,  3.54it/s]


CLIP text embeddings shape: (250853, 512)


In [8]:
# --- Load CLIP image embeddings and fuse with text ---
clip_data = torch.load(f"{DATA_DIR}/beauty/clip_image_embeddings.pt", map_location="cpu")
clip_raw = clip_data["embeddings"]  # (n_dataset_items, 512)
iid_to_idx = clip_data["iid_to_idx"]

n_items = dataset.item_num
fused = clip_text_embs.clone()  # start from text
n_with_image = 0

for internal_id in range(1, n_items):
    token = dataset.id2token(dataset.iid_field, internal_id)
    try:
        iid = int(token)
    except ValueError:
        continue
    dense_idx = iid_to_idx.get(iid)
    if dense_idx is None:
        continue
    img_vec = clip_raw[dense_idx]
    if img_vec.abs().sum() == 0:
        continue
    fused[internal_id] = (clip_text_embs[internal_id] + img_vec) / 2.0
    n_with_image += 1

norms = fused.norm(dim=1, keepdim=True)
norms[norms == 0] = 1.0
fused = fused / norms
fused[0] = 0.0

print(f"Items with both text+image: {n_with_image:,} / {n_items - 1:,} ({100 * n_with_image / max(n_items - 1, 1):.2f}%)")
item_multimodal_embs = fused


/var/folders/vm/77wrgjgj5wzbyghx353b7gym0000gn/T/ipykernel_77755/4236165473.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clip_data = torch.load(f"{DATA_DIR}/beauty/cl

Items with both text+image: 250,822 / 250,852 (99.99%)


## Define BPRClipHybrid

Trainable `user_embedding`/`item_embedding` learn from a standard BPR pairwise loss exactly
like RecBole's own `BPR` model — the CLIP content signal is never part of the loss, only of
the final score. The frozen per-user CLIP profile (mean of each user's *training* items'
multimodal embeddings) is computed once in `__init__`, not accumulated every epoch.

In [9]:
class BPRClipHybrid(GeneralRecommender):
    input_type = InputType.PAIRWISE
    type = ModelType.GENERAL

    def __init__(self, config, dataset, item_content_embeddings: torch.Tensor, alpha: float):
        super().__init__(config, dataset)

        self.alpha = alpha
        self.embedding_size = config["embedding_size"]

        # Trainable collaborative-filtering embeddings (standard BPR)
        self.user_embedding = nn.Embedding(self.n_users, self.embedding_size)
        self.item_embedding = nn.Embedding(self.n_items, self.embedding_size)
        self.loss = BPRLoss()
        self.apply(xavier_normal_initialization)

        # Frozen content signal: CLIP multimodal item embeddings + per-user mean profile
        # computed once from this dataset's (train) interactions.
        content_dim = item_content_embeddings.shape[1]
        self.register_buffer("item_content_embeddings", item_content_embeddings.float())

        train_inter = dataset.inter_feat
        users = train_inter[self.USER_ID]
        items = train_inter[self.ITEM_ID]
        profile_sum = torch.zeros(self.n_users, content_dim)
        profile_cnt = torch.zeros(self.n_users, 1)
        profile_sum.index_add_(0, users, self.item_content_embeddings[items])
        profile_cnt.index_add_(0, users, torch.ones_like(users, dtype=torch.float).unsqueeze(-1))
        self.register_buffer("user_content_profile", profile_sum / profile_cnt.clamp(min=1))

    @staticmethod
    def _row_normalize(x: torch.Tensor) -> torch.Tensor:
        """Z-score each row (e.g. each user's scores across candidate items)."""
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return (x - mean) / (std + 1e-8)

    def calculate_loss(self, interaction):
        user = interaction[self.USER_ID]
        pos_item = interaction[self.ITEM_ID]
        neg_item = interaction[self.NEG_ITEM_ID]

        user_e = self.user_embedding(user)
        pos_e = self.item_embedding(pos_item)
        neg_e = self.item_embedding(neg_item)

        pos_score = (user_e * pos_e).sum(dim=-1)
        neg_score = (user_e * neg_e).sum(dim=-1)
        return self.loss(pos_score, neg_score)

    def predict(self, interaction):
        user = interaction[self.USER_ID]
        item = interaction[self.ITEM_ID]

        bpr_score = (self.user_embedding(user) * self.item_embedding(item)).sum(dim=-1)
        content_score = (self.user_content_profile[user] * self.item_content_embeddings[item]).sum(dim=-1)
        # Single-pair scores can't be row-normalized; blend raw (predict() is unused by the
        # full-sort eval this notebook relies on, kept only for API completeness).
        return self.alpha * bpr_score + (1 - self.alpha) * content_score

    def full_sort_predict(self, interaction):
        user = interaction[self.USER_ID]

        bpr_scores = torch.matmul(self.user_embedding(user), self.item_embedding.weight.t())
        content_scores = torch.matmul(self.user_content_profile[user], self.item_content_embeddings.t())

        blended = self.alpha * self._row_normalize(bpr_scores) + (1 - self.alpha) * self._row_normalize(content_scores)
        return blended.view(-1)

## Train

In [10]:
model: BPRClipHybrid = BPRClipHybrid(
    config, train_data.dataset, item_multimodal_embs, ALPHA
).to(config["device"])
trainer: Trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
23 Jun 12:57    INFO  epoch 0 training [time: 14.05s, train loss: 754.5382]
23 Jun 12:57    INFO  epoch 0 evaluating [time: 29.40s, valid_score: 0.004300]
23 Jun 12:57    INFO  valid result: 
ndcg@20 : 0.0043    recall@20 : 0.0081    mrr@20 : 0.0051
23 Jun 12:57    INFO  Saving current: saved/BPR-Jun-23-2026_12-56-51.pth
23 Jun 12:57    INFO  epoch 1 training [time: 14.13s, train loss: 631.8274]
23 Jun 12:58    INFO  epoch 1 evaluating [time: 27.44s, valid_score: 0.006800]
23 Jun 12:58    INFO  valid result: 
ndcg@20 : 0.0068    recall@20 : 0.0121    mrr@20 : 0.0083
23 Jun 12:58    INFO  Saving current: saved/BPR-Jun-23-2026_12-56-51.pth
23 Jun 12:58    INFO  epoch 2 training [time: 14.44s, train lo

In [11]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0085
Best valid result:
  ndcg@20: 0.0085
  recall@20: 0.0157
  mrr@20: 0.0098


## Evaluate on test set

In [12]:
test_result: dict[str, float] = trainer.evaluate(test_data)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = tor

Test results (Overall):
  ndcg@20: 0.0077
  recall@20: 0.0133
  mrr@20: 0.0094


## Evaluate by user cold/warm, item cold/warm, and the user×item cross-tabulation

The key comparison: plain BPR (`train-bpr2.ipynb`) scores `item-cold` at exactly 0.0000. Does
blending in the frozen CLIP content score move that off zero, the same way BGE-init does for
item-cold (`new_train-bpr-bge-init.ipynb`) — and if so, does it cost anything on `item-warm`,
where the collaborative signal was already doing the real work?

In [13]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    rows.append({
        "Segment": label,
        "Interactions": int(mask.sum()),
        **results,
    })

rows = []

cold_to_label = {0.0: "warm", 1.0: "cold"}

def evaluate_by_column(entity_feat, id_field, inter_id_array, entity_name):
    id_to_cold = dict(zip(
        entity_feat[id_field].numpy(),
        entity_feat["cold"].numpy(),
    ))
    for cold_val, label in cold_to_label.items():
        ids = {eid for eid, c in id_to_cold.items() if c == cold_val}
        mask = np.isin(inter_id_array, list(ids))
        if not mask.any():
            print(f"  {entity_name}-{label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"{entity_name}-{label}")

# Evaluation by user segments
evaluate_by_column(
    dataset.user_feat, dataset.uid_field,
    test_data.dataset.inter_feat[dataset.uid_field].numpy(),
    "user"
)

# Evaluation by item segments
evaluate_by_column(
    dataset.item_feat, dataset.iid_field,
    test_data.dataset.inter_feat[dataset.iid_field].numpy(),
    "item"
)

# Cross-tabulation: user × item segments
uid_to_cold = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["cold"].numpy(),
))
iid_to_cold = dict(zip(
    dataset.item_feat[dataset.iid_field].numpy(),
    dataset.item_feat["cold"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()
iid_array = test_data.dataset.inter_feat[dataset.iid_field].numpy()

for uc_val, uc_label in cold_to_label.items():
    for ic_val, ic_label in cold_to_label.items():
        uc_uids = {uid for uid, c in uid_to_cold.items() if c == uc_val}
        ic_iids = {iid for iid, c in iid_to_cold.items() if c == ic_val}
        mask = np.isin(uid_array, list(uc_uids)) & np.isin(iid_array, list(ic_iids))
        if not mask.any():
            print(f"  user-{uc_label}×item-{ic_label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"user-{uc_label}×item-{ic_label}")

# Display results sorted by NDCG
df_results = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False)
display(df_results)

23 Jun 13:12    INFO  Loading model structure and parameters from saved/BPR-Jun-23-2026_12-56-51.pth
23 Jun 13:12    INFO  Loading model structure and parameters from saved/BPR-Jun-23-2026_12-56-51.pth
23 Jun 13:13    INFO  Loading model structure and parameters from saved/BPR-Jun-23-2026_12-56-51.pth
23 Jun 13:13    INFO  Loading model structure and parameters from saved/BPR-Jun-23-2026_12-56-51.pth
23 Jun 13:13    INFO  Loading model structure and parameters from saved/BPR-Jun-23-2026_12-56-51.pth
23 Jun 13:13    INFO  Loading model structure and parameters from saved/BPR-Jun-23-2026_12-56-51.pth
23 Jun 13:13    INFO  Loading model structure and parameters from saved/BPR-Jun-23-2026_12-56-51.pth
23 Jun 13:14    INFO  Loading model structure and parameters from saved/BPR-Jun-23-2026_12-56-51.pth


,Segment,Interactions,ndcg@20,recall@20,mrr@20
6,user-cold×item-warm,72341,0.0113,0.0208,0.0117
2,item-warm,88077,0.0112,0.0205,0.0120
4,user-warm×item-warm,15736,0.0102,0.0180,0.0142
1,user-cold,150964,0.0079,0.0137,0.0092
0,user-warm,46742,0.0065,0.0105,0.0106
7,user-cold×item-cold,78623,0.0005,0.0008,0.0005
3,item-cold,109629,0.0004,0.0007,0.0005
5,user-warm×item-cold,31006,0.0001,0.0002,0.0000
